# Download and Process MACA Data

This notebook will download MACAv2 (or v3, if access becomes possible!), check for NaNs, and engineer the same features as the PRISM data.

## Step 1: Set up code space

In [39]:
# import libraries

# File management
import glob
import os
import pathlib
from pathlib import Path
import sys # for importing src files

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # may need for api access

# Data Management
import numpy as np
import pandas as pd
import xarray as xr

# Geospatial data
import geopandas as gpd
import rasterio
import rioxarray as rxr

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas

print("Imports Complete!")

Imports Complete!


In [2]:
# Force Jupyter to reload external modules automatically before executing a cell
# I used Gemini to help me set up the connection to my src files

# force reimporting of src scripts
%load_ext autoreload
%autoreload 2

# Find the parent directory of this notebook (the repo root) and add it to Python's search path
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Now Python can see the src folder
# import just specific functions
from src import maca, features

print('src import complete!')

src import complete!


In [3]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

In [4]:
# Import study boundary
mhw_train_path = Path(cleaned_data_dir, 'boundaries', 'mhw_buff_clip_gdf.parquet')
mhw_gdf = gpd.read_parquet(mhw_train_path)

In [5]:
mhw_gdf.total_bounds

array([-113.95841725,   44.34404648, -110.46445972,   46.46949588])

## Step 2: Download MACA data

Missouri Headwaters watershed bounds:
-113.95841725,   44.34404648, -110.46445972,   46.46949588

*Model Ensemble*
- RCP 4.5 (8.5 is no longer realistic)
- Max temp delta JJA, Precip delta DJF

Cool/Wet: CNRM-CM5 (+0.905in, +4.451f)
Cool/Dry: MRI-CGCM3 (+0.155in, +2.279f) - alternate IPSL-CM5B-LR
Warm/Wet: CSIRO-Mk3-6-0 (+0.777in, +7.713f)
Warm/Dry: CanESM2 (+0.361in, +7.528f)

- MAM temp and precip delta
Cool/Wet: CNRM-CM5 (+1.027in, +2.650f)
Cool/Dry: GFDL-ESM2M (+0.086in, +1.302f)
Warm/Wet: MIROC-ESM (+2.234in, +9.141f) - alternate CanESM2 (+1.511in, +5.945f) GO WITH CanESM2
Warm/Dry: BNU-ESM (+0.238in, +7.076f) - go with IPSL-CM5A-MR (+0.147in, +5.537f)

In [6]:
# data dir for MACA data

maca_dir = os.path.join(raw_data_dir, 'maca_data')
os.makedirs(maca_dir, exist_ok = True)

# pattern
maca_pattern = os.path.join(maca_dir, '*.nc')

Need to list and explain model selection here.

In [7]:
# Set desired MACA parameters

models = {'CNRM-CM5'}#, 'GFDL-ESM2M', 'CanESM2', 'IPSL-CM5A-MR'}
variables = {'tasmax'}#, 'tasmin', 'pr'}

In [24]:
# Download MACA data

test = maca.download_maca_data(
    site_gdf=mhw_gdf, site_name='missouri_headwaters',
    models=models, rcp_value='rcp45', variables=variables, start_date= '2074-12-01', end_date='2075-01-01', maca_dir=maca_dir
)
#     mhw_gdf, 'mhw',
#     models, 'rcp45', variables,
#     '2074-10-01', '2075-06-01',
#     maca_dir
# )

Processing: CNRM-CM5 (tasmax): 100%|██████████| 1/1 [00:01<00:00,  1.83s/it]

tasmax | CNRM-CM5 | 2074-12-01 - 2075-01-01 | rcp45 has been downloaded!.


In [ ]:
# check downloaded file
ds_test = xr.open_dataset(test[0])
print(ds_test)
print("\nCoordinates:", list(ds_test.coords))
print("Data variables:", list(ds_test.data_vars))
print("Min:", float(ds_test[list(ds_test.data_vars)[1]].min()))
print("Max:", float(ds_test[list(ds_test.data_vars)[1]].max()))

<xarray.Dataset> Size: 543kB
Dimensions:      (time: 32, lat: 51, lon: 83)
Coordinates:
  * time         (time) datetime64[ns] 256B 2074-12-01 2074-12-02 ... 2075-01-01
  * lat          (lat) float64 408B 44.35 44.4 44.44 44.48 ... 46.35 46.4 46.44
  * lon          (lon) float64 664B -113.9 -113.9 -113.8 ... -110.5 -110.5
    spatial_ref  int64 8B ...
Data variables:
    crs          int64 8B ...
    tasmax       (time, lat, lon) float32 542kB ...
Attributes: (12/46)
    description:                     Multivariate Adaptive Constructed Analog...
    id:                              MACAv2-METDATA
    naming_authority:                edu.uidaho.reacch
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    Metadata_Link:                   
    cdm_data_type:                   GRID
    ...                              ...
    contributor_role:                Postdoctoral Fellow
    publisher_name:                  
    publisher_email:                 
    publisher_url:

In [8]:
# set up params for full download

models = {'CNRM-CM5', 'GFDL-ESM2M', 'CanESM2', 'IPSL-CM5A-MR'}
variables = {'tasmax', 'tasmin', 'pr'}

In [10]:
# complete download
maca_paths = maca.download_maca_data(
    # set site gdf and name
    site_gdf=mhw_gdf, site_name='missouri_headwaters',
    # set models, rcp value, and variables of interest
    models=models, rcp_value='rcp45', variables=variables, 
    # set start date and end date
    start_date= '2045-10-01', end_date='2065-01-01', 
    # set dir
    maca_dir=maca_dir
)

Processing: GFDL-ESM2M (pr):  17%|█▋        | 2/12 [00:00<00:00, 107.16it/s]    

tasmin | GFDL-ESM2M | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmax | GFDL-ESM2M | 2045-10-01 - 2065-01-01 | rcp45 | already exists.


Processing: CanESM2 (tasmin):  25%|██▌       | 3/12 [01:34<04:42, 31.33s/it]

pr | GFDL-ESM2M | 2045-10-01 - 2065-01-01 | rcp45 | has been downloaded!.


Processing: CanESM2 (tasmax):  33%|███▎      | 4/12 [02:49<06:05, 45.67s/it]

...

tasmin | CNRM-CM5 | 2045-10-01 - 2065-01-01 | rcp45 | has been downloaded!.


Processing: CNRM-CM5 (pr):  92%|█████████▏| 11/12 [11:52<01:13, 73.58s/it]    

tasmax | CNRM-CM5 | 2045-10-01 - 2065-01-01 | rcp45 | has been downloaded!.


Processing: CNRM-CM5 (pr): 100%|██████████| 12/12 [13:09<00:00, 65.75s/it]

pr | CNRM-CM5 | 2045-10-01 - 2065-01-01 | rcp45 | has been downloaded!.


In [ ]:
# check the paths
maca_paths

['C:\\Users\\raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_tasmin_GFDL-ESM2M_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_tasmax_GFDL-ESM2M_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_pr_GFDL-ESM2M_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_tasmin_CanESM2_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_tasmax_CanESM2_missouri_headwaters_rcp45_2045-10-01_2065-01-

## Step 3: Feature Engineering and Preparation for ML Application

In [ ]:
# load models into datasets and perform feature engineering
# I'm using a dict here to make this code more DRY and flexible - it relies on the 'models' variable above

# initialize dict
datasets = {}

# loop through model names and load data into lazy datasets
for model in models:
    # Print status update
    print(f"Loading and processing {model}...")

    # make search pattern and get paths
    pattern = os.path.join(maca_dir, f'maca_*_{model}_*.nc')
    file_paths = glob.glob(pattern)

    # open model ds and combine tasmin, tasmax, and pr
    model_ds = xr.open_mfdataset(file_paths, combine = 'by_coords')

    # standardize variable names
    rename_map = {k: v for k, v in {'pr': 'ppt', 'tasmax': 'tmax', 'tasmin': 'tmin'}.items() if k in model_ds}
    if rename_map:
        model_ds = model_ds.rename(rename_map)

    # perform feature engineering
    model_ds = features.engineer_all_features(model_ds)

    # append to dict
    datasets[model] = model_ds

Loading and processing GFDL-ESM2M...


C:\Users\raini\AppData\Local\Temp\ipykernel_17404\797921138.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(file_paths, combine = 'by_coords')


Loading and processing CanESM2...


C:\Users\raini\AppData\Local\Temp\ipykernel_17404\797921138.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(file_paths, combine = 'by_coords')


Loading and processing IPSL-CM5A-MR...


C:\Users\raini\AppData\Local\Temp\ipykernel_17404\797921138.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(file_paths, combine = 'by_coords')


Loading and processing CNRM-CM5...


C:\Users\raini\AppData\Local\Temp\ipykernel_17404\797921138.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(file_paths, combine = 'by_coords')


In [ ]:
# check out a model ds
datasets['GFDL-ESM2M']

<xarray.Dataset> Size: 962MB
Dimensions:                   (time: 4734, lat: 51, lon: 83)
Coordinates:
  * lat                       (lat) float64 408B 44.35 44.4 44.44 ... 46.4 46.44
  * time                      (time) datetime64[ns] 38kB 2045-10-01 ... 2065-...
  * lon                       (lon) float64 664B -113.9 -113.9 ... -110.5 -110.5
    spatial_ref               int64 8B 0
    water_year                (time) int64 38kB 2046 2046 2046 ... 2065 2065
    dowy                      (time) int64 38kB 1 2 3 4 5 6 ... 89 90 91 92 93
Data variables:
    ppt                       (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    tmax                      (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    tmin                      (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    tmean_c                   (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    wy_cumul_ppt_mm           (time, lat, lon) float32 80MB dask.array<chunksize=(244, 51, 83), meta=np.ndarray>
    tmin_roll3_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 26, 42), meta=np.ndarray>
    tmax_roll3_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 26, 42), meta=np.ndarray>
    ppt_roll3_mm              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 26, 42), meta=np.ndarray>
    tmin_roll7_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 17, 28), meta=np.ndarray>
    tmax_roll7_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 17, 28), meta=np.ndarray>
    ppt_roll7_mm              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 17, 28), meta=np.ndarray>
    wy_cumul_pos_degree_days  (time, lat, lon) float32 80MB dask.array<chunksize=(244, 51, 83), meta=np.ndarray>
Attributes: (12/46)
    description:                     Multivariate Adaptive Constructed Analog...
    id:                              MACAv2-METDATA
    naming_authority:                edu.uidaho.reacch
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    Metadata_Link:                   
    cdm_data_type:                   GRID
    ...                              ...
    contributor_role:                Postdoctoral Fellow
    publisher_name:                  
    publisher_email:                 
    publisher_url:                   
    license:                         Creative Commons CC0 1.0 Universal Dedic...
    coordinate_system:               WGS84,EPSG:4326

In [41]:
# import DEM datasets

# set topo dir
topo_rsmp_dir = Path(cleaned_data_dir, 'topo')

# paths
elev_4km_path = Path(topo_rsmp_dir, 'mhw_elevation_4km.tif')
slope_4km_path = Path(topo_rsmp_dir, 'mhw_slope_4km.tif')
aspect_4km_path = Path(topo_rsmp_dir, 'mhw_aspect_4km.tif')
aspect_north_4km_path = Path(topo_rsmp_dir, 'mhw_northness_4km.tif')
aspect_east_4km_path = Path(topo_rsmp_dir, 'mhw_eastness_4km.tif')

# import
mhw_elev_4km_da = rxr.open_rasterio(elev_4km_path)
mhw_slope_4km_da = rxr.open_rasterio(slope_4km_path)
mhw_aspect_4km_da = rxr.open_rasterio(aspect_4km_path)
mhw_aspect_4km_north_da = rxr.open_rasterio(aspect_north_4km_path)
mhw_aspect_4km_east_da = rxr.open_rasterio(aspect_east_4km_path)

# merge into one dataset
dem_ds = xr.Dataset({
    'elevation': mhw_elev_4km_da,
    'slope': mhw_slope_4km_da,
    'aspect': mhw_aspect_4km_da,
    'aspect_north': mhw_aspect_4km_north_da,
    'aspect_east': mhw_aspect_4km_east_da
})

# make sure dataset is in ESPG: 4326
if dem_ds.rio.crs.to_epsg() != 4326:
    dem_ds = dem_ds.rio.reproject("EPSG:4326")

In [25]:
dem_ds

<xarray.Dataset> Size: 77kB
Dimensions:       (band: 1, x: 83, y: 51)
Coordinates:
  * band          (band) int64 8B 1
  * x             (x) float64 664B -113.9 -113.9 -113.8 ... -110.6 -110.5 -110.5
  * y             (y) float64 408B 46.46 46.42 46.38 46.33 ... 44.46 44.42 44.38
    spatial_ref   int64 8B 0
Data variables:
    elevation     (band, y, x) int16 8kB ...
    slope         (band, y, x) float32 17kB ...
    aspect        (band, y, x) float32 17kB ...
    aspect_north  (band, y, x) float32 17kB ...
    aspect_east   (band, y, x) float32 17kB ...

In [29]:
# check MACA and DEM lat/lon alignment

# lon
print('DEM lon', dem_ds.x[0:5])
print('MACA lon:', datasets['GFDL-ESM2M'].lon[0:5])

# lat
print('DEM lat', dem_ds.y[0:5])
print('MACA lat:', datasets['GFDL-ESM2M'].lat[0:5])

DEM lon <xarray.DataArray 'x' (x: 5)> Size: 40B
array([-113.916669, -113.875002, -113.833336, -113.791669, -113.750002])
Coordinates:
  * x            (x) float64 40B -113.9 -113.9 -113.8 -113.8 -113.8
    spatial_ref  int64 8B 0
MACA lon: <xarray.DataArray 'lon' (lon: 5)> Size: 40B
array([-113.897339, -113.855682, -113.813995, -113.772339, -113.730682])
Coordinates:
  * lon          (lon) float64 40B -113.9 -113.9 -113.8 -113.8 -113.7
    spatial_ref  int64 8B 0
Attributes:
    axis:           X
    long_name:      longitude
    standard_name:  longitude
    units:          degrees_east
DEM lat <xarray.DataArray 'y' (y: 5)> Size: 40B
array([46.458337, 46.41667 , 46.375004, 46.333337, 46.29167 ])
Coordinates:
  * y            (y) float64 40B 46.46 46.42 46.38 46.33 46.29
    spatial_ref  int64 8B 0
MACA lat: <xarray.DataArray 'lat' (lat: 5)> Size: 40B
array([44.354435, 44.396103, 44.437767, 44.479435, 44.521099])
Coordinates:
  * lat          (lat) float64 40B 44.35 44.4 44.44 44.48 44

Need to flip DEM or MACA lat, and match the grids up. Will probably use reproject match there.

In [60]:
# align DEM and MACA grid

# drop band and mask elevation NaNs
dem_clean_ds = dem_ds.squeeze('band', drop = True)
dem_clean_ds['elevation'] = dem_clean_ds['elevation'].where(dem_clean_ds['elevation'] > -1000)

# set maca template
maca_template = datasets['GFDL-ESM2M']['tmin'].isel(time=0)

# merge DEM and MACA grids
dem_align_ds = dem_clean_ds.rio.reproject_match(
    # match to MACA grid
    maca_template,
    # resample w/ bilinear method
    resampling = rasterio.enums.Resampling.bilinear)

# rename
dem_rnm_ds = dem_clean_ds.rename({'x': 'lon', 'y': 'lat'})

In [ ]:
# drop spatial ref and NaNs; flatten dem ds
dem_df = dem_rnm_ds.drop_vars('spatial_ref').to_dataframe().dropna(subset=['elevation'])

In [ ]:
# convert MACA ds to df

In [ ]:
# reset index

In [ ]:
# rename cols

# reorder cols

In [ ]:
# drop NaNs and check

## Step 4: Save cleaned and engineered MACA data for ML usage